In [15]:
# Load Doc

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

loader=PyPDFLoader("10 RAG Architectures.pdf")
documents=loader.load()

splitter= RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks=splitter.split_documents(documents)

#load Embedding model
embeddings=HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Embed all Chunks and store in FAISS
vectorstore=FAISS.from_documents(chunks,embeddings)

# Save to disk so you dont Re-embed every time
vectorstore.save_local("faiss_index")

# Load later
vectorstore= FAISS.load_local("faiss_index", embeddings, allow_dangerous_deserialization=True)

# Create retriever from vectorstore
retriver=vectorstore.as_retriever(
    search_kwargs={"k":3} # Return top three most similar chunks
)

# Retrieve for a query
query= "name some RAG architecture"
relevant_chunks= retriver.invoke(query)

# See what was retrieved
for chunk in relevant_chunks:
    print(chunk.page_content)
    print("---")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6439.60it/s]


10  RAG  Architectures    1.  Standard  RAG  
The  baseline  Retrieval-Augmented  Generation  setup.  
A  user  query  is  embedded,  relevant  documents  are  retrieved  from  a  vector  
database,
 
and
 
the
 
LLM
 
generates
 
an
 
answer
 
grounded
 
in
 
that
 
context.
 
Example  
User:  “What  are  the  side  effects  of  ibuprofen?”  
 
System:
 
●  Retrieves  medical  documents  ●  Feeds  them  to  the  LLM  ●  Generates  a  grounded  answer  
Usage
---
7.  Attention-based  RAG  
Uses  attention  mechanisms  to  prioritize  the  most  relevant  parts  of  retrieved  
documents.
 
 
Instead
 
of
 
treating
 
all
 
retrieved
 
chunks
 
equally,
 
the
 
model:
 
●  assigns  weights  to  different  passages  ●  focuses  on  high-signal  content  during  generation  
 
Example  
User:  “Causes  of  climate  change?”  
 
System:
---
Optimizes  RAG  pipelines  under  cost  constraints  (tokens,  API  calls,  latency).  
 
The
 
system
 
dynamically:
 
●  limits  retrieval  depth  ● 

# Final step
Combine retrieve chunks + User question into a prompt, sent to LLM, get grounded answer.

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
    api_key="your_actual_api_key"
)

def ask_rag(query):
    # Retrieve chunks
    chunks = retriver.invoke(query)
    context = "\n".join([c.page_content for c in chunks])
    
    # Build prompt manually
    prompt = f"""
You are an expert on RAG architectures.
Use ONLY the context below to answer.
If not in context, say "I don't know."

Context: {context}
Question: {query}
Answer:
"""
    # Ask LLM
    response = llm.invoke(prompt)
    return response.content

# Test
print(ask_rag("What is Standard RAG?"))
print(ask_rag("Name all 10 RAG architectures"))

The Standard RAG is the baseline Retrieval-Augmented Generation setup. It works by: 
1. Embedding a user query
2. Retrieving relevant documents from a vector database
3. Feeding them to the LLM (Large Language Model)
4. Generating an answer grounded in that context.
I don't know. The context only mentions 3 RAG architectures: 

1. Standard RAG
7. Attention-based RAG
and a third one that is not numbered but seems to be related to cost optimization, but it is not explicitly named. The other 7 architectures are not mentioned in the context.


# Key things in the prompt
→"Use ONLY the context below" — grounds the LLM, prevents hallucination.

→"If not in context say I don't know" — prevents making things up.

→{context} gets filled with retrieved chunks automatically.

→{question} gets filled with user's query.